# r2-001 Problem 1 — Solution (programming Parts 1.11–1.13)

Reference solution for Parts 1.11–1.13 of `problems/p01.ipynb`.
Theory parts are in `answers.md`.
The cell below is the problem notebook's supplied setup cell, unchanged.

In [ ]:
import importlib.util
import json
import math
import os
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261101
torch.set_num_threads(1)


def find_data_dir() -> Path:
    """Locate r2-001's data/ folder from the notebook's working directory."""
    candidates = [Path("../data"), Path("data"), Path("mocktests/r2-001/data"), Path("book2/mocktests/r2-001/data")]
    env_root = os.environ.get("USAAIO_BOOK_ROOT")
    if env_root:
        candidates.insert(0, Path(env_root) / "mocktests" / "r2-001" / "data")
    for path in candidates:
        if (path / "gen_r2_001.py").is_file():
            return path.resolve()
    raise FileNotFoundError("cannot find mocktests/r2-001/data; run from problems/ or set USAAIO_BOOK_ROOT")


DATA_DIR = find_data_dir()

corpus = json.loads((DATA_DIR / "p01_corpus.json").read_text(encoding="utf-8"))
train_tokens = torch.tensor(corpus["train"], dtype=torch.int64)
val_tokens = torch.tensor(corpus["val"], dtype=torch.int64)
VOCAB_SIZE, D_MODEL, N_HEADS, N_LAYERS, D_FF, MAX_LEN = 24, 32, 4, 2, 64, 12
print(train_tokens.shape, val_tokens.shape)

## Part 1.11 — causal mask and masked softmax

In [ ]:
def causal_mask(n):
    """(n, n) bool; entry [i, j] is True (allowed) exactly when j <= i."""
    return torch.tril(torch.ones(n, n, dtype=torch.bool))


def masked_softmax(scores, allowed):
    """Softmax over the last axis after replacing forbidden scores by -inf."""
    allowed = torch.as_tensor(allowed, dtype=torch.bool, device=scores.device)
    allowed_b = allowed.expand_as(scores)
    if not bool(allowed_b.any(dim=-1).all()):
        raise ValueError("masked_softmax: some row has no allowed entry")
    masked = scores.masked_fill(~allowed_b, float("-inf"))
    return torch.softmax(masked, dim=-1)


# Probe.
S = torch.log(torch.arange(1, 5, dtype=torch.float64)).repeat(4, 1)       # S[i, j] = ln(j + 1)
Vcol = torch.tensor([[1.0], [2.0], [3.0], [4.0]], dtype=torch.float64)
mask4 = causal_mask(4)
W = masked_softmax(S, mask4)
out = W @ Vcol
S_mod = S.clone()
S_mod[0, 3] = 100.0
W_mod = masked_softmax(S_mod, mask4)
out_mod = W_mod @ Vcol
print("out[:, 0] =", out[:, 0])

assert mask4.shape == (4, 4) and mask4.dtype == torch.bool
assert torch.allclose(W.sum(-1), torch.ones(4, dtype=torch.float64), atol=1e-12, rtol=0)
assert bool((W[~mask4] == 0).all()) and bool((W_mod[~mask4] == 0).all())
assert torch.equal(out, out_mod)
try:
    masked_softmax(S, torch.tensor([[True, False, False, False], [False] * 4, [True] * 4, [True] * 4]))
    raise AssertionError("expected ValueError for an all-False row")
except ValueError:
    pass
# Row i has weights proportional to (1, ..., i+1), so out[i] = sum k^2 / sum k = (2i+3)/3.
assert torch.allclose(out[:, 0], torch.tensor([1.0, 5 / 3, 7 / 3, 3.0], dtype=torch.float64), atol=1e-12, rtol=0)
print(f"Part 1.11 answer: out[2, 0] = {float(out[2, 0]):.6f}")

In [ ]:
ANSWER = 2.333333
assert abs(ANSWER - float(out[2, 0])) <= 1e-6


## Part 1.12 — multi-head causal self-attention

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")
        self.d_model, self.n_heads, self.d_h = d_model, n_heads, d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.o_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, n, _ = x.shape
        if n > MAX_LEN:
            raise ValueError("context longer than 12")

        def split(t):
            return t.view(B, n, self.n_heads, self.d_h).transpose(1, 2)        # (B, h, n, d_h)

        q, k, v = split(self.q_proj(x)), split(self.k_proj(x)), split(self.v_proj(x))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_h)                 # (B, h, n, n)
        weights = masked_softmax(scores, causal_mask(n).to(x.device))
        heads = weights @ v                                                    # (B, h, n, d_h)
        merged = heads.transpose(1, 2).contiguous().view(B, n, self.d_model)
        return self.o_proj(merged)


torch.manual_seed(SEED)
attn = CausalSelfAttention(32, 4)
x = torch.randn(2, 12, 32, generator=torch.Generator().manual_seed(SEED))
out = attn(x)
x2 = x.clone()
x2[:, 7:] += 1.0
out2 = attn(x2)

n_params = sum(p.numel() for p in attn.parameters())
print("n_params =", n_params)
assert n_params == 4 * 32 * 32 + 4 * 32 == 4224          # Part 1.4(a) at d = 32
assert out.shape == (2, 12, 32)
assert (out2[:, :7] - out[:, :7]).abs().max() == 0
assert (out2[:, 7:] - out[:, 7:]).abs().max() > 0
try:
    CausalSelfAttention(30, 4)
    raise AssertionError("expected ValueError")
except ValueError:
    pass
probe_value = float(out.detach().abs().mean())   # = float(out.abs().mean()), detached to avoid a warning
print(f"Part 1.12 answer: probe_value = {probe_value:.6f}")

In [ ]:
ANSWER = 0.167092
assert abs(ANSWER - probe_value) <= 1e-4


## Part 1.13 — the full model, training, and perplexity

In [ ]:
def sinusoidal_table(L, d):
    p = torch.arange(L, dtype=torch.float64)[:, None]
    i = torch.arange(d // 2, dtype=torch.float64)[None, :]
    omega = 10000.0 ** (-2.0 * i / d)
    table = torch.zeros(L, d, dtype=torch.float64)
    table[:, 0::2] = torch.sin(p * omega)
    table[:, 1::2] = torch.cos(p * omega)
    return table.float()


def shift_for_causal_lm(tokens):
    return tokens[:, :-1], tokens[:, 1:]


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(D_MODEL)
        self.attn = CausalSelfAttention(D_MODEL, N_HEADS)
        self.ln2 = nn.LayerNorm(D_MODEL)
        self.ff1 = nn.Linear(D_MODEL, D_FF)
        self.ff2 = nn.Linear(D_FF, D_MODEL)

    def forward(self, x):
        y = x + self.attn(self.ln1(x))
        return y + self.ff2(F.relu(self.ff1(self.ln2(y))))


class TinyCausalLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.register_buffer("pe", sinusoidal_table(MAX_LEN, D_MODEL))
        self.blocks = nn.ModuleList([Block() for _ in range(N_LAYERS)])
        self.ln_f = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, VOCAB_SIZE)

    def forward(self, ids):
        n = ids.shape[1]
        h = self.tok(ids) + self.pe[:n]
        for block in self.blocks:
            h = block(h)
        return self.head(self.ln_f(h))


def per_position_ce(model, tokens):
    inputs, targets = shift_for_causal_lm(tokens)
    logits = model(inputs)
    ce = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1), reduction="none")
    return logits, ce.view(targets.shape)                                       # (B, 12)


torch.manual_seed(SEED)
model = TinyCausalLM()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)

model.eval()
with torch.no_grad():
    _, ce_before = per_position_ce(model, val_tokens)
val_ppl_before = math.exp(float(ce_before.mean()))

inputs, targets = shift_for_causal_lm(train_tokens)
losses = []
model.train()
for _ in range(150):
    optimizer.zero_grad(set_to_none=True)
    logits = model(inputs)
    loss = F.cross_entropy(logits.reshape(-1, 24), targets.reshape(-1))
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

model.eval()
with torch.no_grad():
    val_logits, ce_after = per_position_ce(model, val_tokens)
val_ppl_after = math.exp(float(ce_after.mean()))
per_pos_ppl = torch.exp(ce_after.mean(dim=0))

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("trainable parameters:", n_trainable)
print("first / last loss:", round(losses[0], 4), round(losses[-1], 4))
print(f"val_ppl_before = {val_ppl_before:.3f}")
print("per-position val PPL:", [round(v, 3) for v in per_pos_ppl.tolist()])

assert n_trainable == 18712                                         # Part 1.5
assert all(p is not model.pe for p in model.parameters()) and "pe" not in dict(model.named_parameters())
assert bool((model.pe[0, 0::2] == 0).all()) and bool((model.pe[0, 1::2] == 1).all())
assert val_logits.shape == (64, 12, 24)
assert len(losses) == 150 and losses[-1] < 0.25 * losses[0]
assert val_ppl_after < val_ppl_before
assert 1.05 < val_ppl_after < 1.20
assert int(per_pos_ppl.argmax()) == 0
print(f"Part 1.13 answer: val_ppl_after = {val_ppl_after:.3f}")

In [ ]:
ANSWER = 1.132
assert abs(ANSWER - val_ppl_after) <= 0.03


## Explanation for Part 1.13

At output position 0 the model has seen only $t_0$, and the step $k\in\{1,2,3\}$ is independent of $t_0$, so even the rule-aware predictor of Part 1.14(a) can do no better than probability $1/3$ there (per-position perplexity $3$), while from position 1 on the step is determined by $t_0,t_1$ and the next token is predictable with probability near $1$.
Hence position 0 carries almost all of the irreducible loss, and the overall perplexity approaches $3^{1/12}\approx1.096$ from above.

### Answer check

In [ ]:
print(f"1.11: {float(masked_softmax(S, causal_mask(4))[2] @ Vcol[:, 0]):.6f}")
print(f"1.12: {probe_value:.6f}")
print(f"1.13: {val_ppl_after:.3f}")
assert abs(float(masked_softmax(S, causal_mask(4))[2] @ Vcol[:, 0]) - 7 / 3) < 1e-12
assert val_ppl_after >= 3 ** (1 / 12) - 0.02     # cannot beat the Bayes floor by much on 64 rows